# Loading Monthly Data to a Database

While it may seem unnecessary, one reason this is potentially helpful is the total size of the data.
It might be prohibitive to actually load everything into memory and as such it may be a better idea to upload data to a database and download specific parts as necessary for analysis.

In [33]:
import pandas as pd
import numpy as np
import os
import datetime as dt
import json
import pymongo
import sqlalchemy

The relational schema will keep numeric data as-is but link them to their value labels through foreign keys.
Retrieve value labels from MongoDB.

In [7]:
_mongo_details = json.load(open('.mongo.json'))
_mongo_url = 'mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority'.format(*_mongo_details.values())
client = pymongo.mongo_client.MongoClient(_mongo_url, server_api=pymongo.server_api.ServerApi('1'))
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)
lfs_db = client['lfs']
lfs_var_labs = lfs_db['variable_labels']
varlabs = lfs_var_labs.find_one()['labels']
lfs_val_labs = lfs_db['value_labels']
vallabs = lfs_val_labs.find_one()['labels']
client.close()

Pinged your deployment. You successfully connected to MongoDB!


Invert the order of values and labels in the value labels document.

In [8]:
vallabs = {v.lower(): {i: s for s, i in vl.items()} for v, vl in vallabs.items()}
next(iter(vallabs)), vallabs[next(iter(vallabs))]

('survmnth',
 {1: 'January',
  2: 'February',
  3: 'March',
  4: 'April',
  5: 'May',
  6: 'June',
  7: 'July',
  8: 'August',
  9: 'September',
  10: 'October',
  11: 'November',
  12: 'December'})

## Setting Up the Database Schema

Before accessing the relational database, a few things will be helpful.
First is the `MetaData` object.

In [9]:
lfs_metadata = sqlalchemy.MetaData()

Second is a function to create a `Table` given a `DataFrame`.

In [10]:
def df_to_table(df:pd.DataFrame, id_cols:list[str], metadata:sqlalchemy.MetaData, table_name:str, for_ids:dict=None):
    meta_pd = df.convert_dtypes(convert_integer=True).dtypes
    meta_pd = meta_pd.map({
        pd.Int64Dtype(): sqlalchemy.Integer,
        pd.Float64Dtype(): sqlalchemy.Float,
        pd.StringDtype(): sqlalchemy.String
    })
    meta_sql = sqlalchemy.Table(table_name, metadata)
    for v, t in zip(meta_pd.index, meta_pd):
        if v in id_cols:
            meta_sql.append_column(column=sqlalchemy.Column(v, t, primary_key=True))
        elif for_ids is not None and v in for_ids:
            meta_sql.append_column(column=sqlalchemy.Column(v, t, sqlalchemy.ForeignKey(for_ids[v])))
        else:
            meta_sql.append_column(column=sqlalchemy.Column(v, t))
    return meta_sql

Test the function

In [11]:
df = pd.DataFrame.from_dict({'label': vallabs[next(iter(vallabs))]}).rename_axis(index='value').reset_index(drop=False)
df_to_table(df, ['value'], lfs_metadata, next(iter(vallabs)))

Table('survmnth', MetaData(), Column('value', Integer(), table=<survmnth>, primary_key=True, nullable=False), Column('label', String(), table=<survmnth>), schema=None)

Keeping that example table in memory will be troublesome.

In [12]:
lfs_metadata.clear()

Testing with a LFS data set.

In [13]:
df = pd.read_csv('data/lfs-2022-01.tab', sep='\t')
df = df.rename(columns=dict(zip(df.columns, df.columns.str.lower())))
df_to_table(df, ['survyear','survmnth','rec_num'], lfs_metadata, 'lfs')

Table('lfs', MetaData(), Column('rec_num', Integer(), table=<lfs>, primary_key=True, nullable=False), Column('survyear', Integer(), table=<lfs>, primary_key=True, nullable=False), Column('survmnth', Integer(), table=<lfs>, primary_key=True, nullable=False), Column('lfsstat', Integer(), table=<lfs>), Column('prov', Integer(), table=<lfs>), Column('cma', Integer(), table=<lfs>), Column('age_12', Integer(), table=<lfs>), Column('age_6', Integer(), table=<lfs>), Column('sex', Integer(), table=<lfs>), Column('marstat', Integer(), table=<lfs>), Column('educ', Integer(), table=<lfs>), Column('mjh', Integer(), table=<lfs>), Column('everwork', Integer(), table=<lfs>), Column('ftptlast', Integer(), table=<lfs>), Column('cowmain', Integer(), table=<lfs>), Column('immig', Integer(), table=<lfs>), Column('naics_21', Integer(), table=<lfs>), Column('noc_10', Integer(), table=<lfs>), Column('noc_43', Integer(), table=<lfs>), Column('yabsent', Integer(), table=<lfs>), Column('wksaway', Integer(), tabl

Keeping this table in memory will be troublesome.

In [14]:
lfs_metadata.clear()

Define the entire MetaData of the LFS database, starting first with the value label tables then the main LFS data table.

In [15]:
for_ids = dict()
for var, vl in vallabs.items():
    vl = {'label': vl}
    df = pd.DataFrame().from_dict(vl)
    df = df.rename_axis(index='value').reset_index(drop=False)
    df_to_table(df, ['value'], lfs_metadata, var)
    for_ids[var] = '{}.{}'.format(var, lfs_metadata.tables[var].primary_key.columns[0].name)
df = pd.read_csv('data/lfs-2022-01.tab', sep='\t')
df = df.rename(columns=dict(zip(df.columns, df.columns.str.lower())))
df_to_table(df, ['survyear','survmnth','rec_num'], lfs_metadata, 'lfs', for_ids)

Table('lfs', MetaData(), Column('rec_num', Integer(), table=<lfs>, primary_key=True, nullable=False), Column('survyear', Integer(), table=<lfs>, primary_key=True, nullable=False), Column('survmnth', Integer(), table=<lfs>, primary_key=True, nullable=False), Column('lfsstat', Integer(), ForeignKey('lfsstat.value'), table=<lfs>), Column('prov', Integer(), ForeignKey('prov.value'), table=<lfs>), Column('cma', Integer(), ForeignKey('cma.value'), table=<lfs>), Column('age_12', Integer(), ForeignKey('age_12.value'), table=<lfs>), Column('age_6', Integer(), ForeignKey('age_6.value'), table=<lfs>), Column('sex', Integer(), ForeignKey('sex.value'), table=<lfs>), Column('marstat', Integer(), ForeignKey('marstat.value'), table=<lfs>), Column('educ', Integer(), ForeignKey('educ.value'), table=<lfs>), Column('mjh', Integer(), ForeignKey('mjh.value'), table=<lfs>), Column('everwork', Integer(), ForeignKey('everwork.value'), table=<lfs>), Column('ftptlast', Integer(), ForeignKey('ftptlast.value'), ta

## Setting Up the Database

Using the defined schema, we can now create the database.

In [16]:
lfs_eng = sqlalchemy.create_engine(sqlalchemy.URL.create(drivername='postgresql+psycopg', **json.load(open('.psql.json', 'rt')), database='lfs'))
lfs_metadata.create_all(lfs_eng)

Verify all Tables are created.

In [17]:
new_metadata_obj = sqlalchemy.MetaData()
new_metadata_obj.reflect(bind=lfs_eng)
for t in new_metadata_obj.tables:
    print('{: >16s}:'.format(t), [c.name for c in new_metadata_obj.tables[t].columns])

        survmnth: ['value', 'label']
         lfsstat: ['value', 'label']
             lfs: ['rec_num', 'survyear', 'survmnth', 'lfsstat', 'prov', 'cma', 'age_12', 'age_6', 'sex', 'marstat', 'educ', 'mjh', 'everwork', 'ftptlast', 'cowmain', 'immig', 'naics_21', 'noc_10', 'noc_43', 'yabsent', 'wksaway', 'payaway', 'uhrsmain', 'ahrsmain', 'ftptmain', 'utothrs', 'atothrs', 'hrsaway', 'yaway', 'paidot', 'unpaidot', 'xtrahrs', 'whypt', 'tenure', 'prevten', 'hrlyearn', 'union', 'permtemp', 'estsize', 'firmsize', 'durunemp', 'flowunem', 'unemftpt', 'whylefto', 'whyleftn', 'durjless', 'availabl', 'lkpubag', 'lkemploy', 'lkrels', 'lkatads', 'lkansads', 'lkothern', 'prioract', 'ynolook', 'tlolook', 'schooln', 'efamtype', 'agyownk', 'finalwt']
          age_12: ['value', 'label']
           age_6: ['value', 'label']
         agyownk: ['value', 'label']
        availabl: ['value', 'label']
             cma: ['value', 'label']
         cowmain: ['value', 'label']
            educ: ['value', 'label'

Close the extra metadata object.

In [18]:
new_metadata_obj.clear()
del new_metadata_obj

## Populating the Database

Populating the database is straightforward with `SQLAlchemy`'s API.
We tackle the value labels first because they are foreign keys to the main table.

In [20]:
for var, vl in vallabs.items():
    vl = {'label': vl}
    df = pd.DataFrame().from_dict(vl)
    df = df.rename_axis(index='value').reset_index(drop=False)
    df = df.to_dict(orient='records')
    with lfs_eng.connect() as conn:
        result = conn.execute(sqlalchemy.insert(lfs_metadata.tables[var]).values(df).returning(*list(lfs_metadata.tables[var].primary_key.columns)))
        conn.commit()
        print('{: >16s}: {} rows'.format(var, result.rowcount))

        survmnth: 12 rows
         lfsstat: 4 rows
            prov: 10 rows
             cma: 10 rows
          age_12: 12 rows
           age_6: 6 rows
             sex: 2 rows
         marstat: 6 rows
            educ: 7 rows
             mjh: 2 rows
        everwork: 3 rows
        ftptlast: 2 rows
         cowmain: 7 rows
           immig: 3 rows
        naics_21: 21 rows
          noc_10: 10 rows
          noc_43: 43 rows
         yabsent: 4 rows
         payaway: 2 rows
        ftptmain: 2 rows
           yaway: 5 rows
           whypt: 8 rows
           union: 3 rows
        permtemp: 4 rows
         estsize: 4 rows
        firmsize: 4 rows
        flowunem: 8 rows
        unemftpt: 3 rows
        whylefto: 6 rows
        whyleftn: 14 rows
        availabl: 2 rows
         lkpubag: 1 rows
        lkemploy: 1 rows
          lkrels: 1 rows
         lkatads: 1 rows
        lkansads: 1 rows
        lkothern: 1 rows
        prioract: 4 rows
         ynolook: 7 rows
         tlolook:

We now handle the main table.
The records to be inserted are all records from Jan 2006 through Apr 2023.

In [46]:
data_files = [d.path for d in os.scandir('data')]
data_files = list(filter(lambda s: s.split('/')[-1].startswith('lfs'), data_files))
data_files = sorted(data_files, key=lambda s: tuple(map(int, s.split('/')[-1].rsplit('.', maxsplit=1)[0].split('-')[1:])))
data_files = list(filter(lambda s: int(s.split('/')[-1].split('-')[1])>=2006, data_files))
chunksize = 1_000
for f in data_files:
    dt_start = dt.datetime.now()
    df = pd.read_csv(f, sep='\t', encoding='utf-8')
    df = df.rename(columns=dict(zip(df.columns, df.columns.str.lower().str.strip())))
    df = df.convert_dtypes(convert_integer=True)
    df = df.to_dict(orient='records')
    inserted_rows = 0
    for i in range(0,len(df)+1,chunksize):
        with lfs_eng.connect() as conn:
            result = conn.execute(sqlalchemy.insert(lfs_metadata.tables['lfs']).values(df[i:i+chunksize]).returning(*list(lfs_metadata.tables['lfs'].primary_key.columns)))
            inserted_rows += result.rowcount
    print('{: >16s}: {} rows after {}'.format(f.split('/')[-1].rsplit('.', maxsplit=1)[0], inserted_rows, dt.datetime.now() - dt_start))

     lfs-2006-01: 100744 rows after 0:02:09.489119
     lfs-2006-02: 100674 rows after 0:02:06.769357
     lfs-2006-03: 100048 rows after 0:01:57.163288
     lfs-2006-04: 100786 rows after 0:01:56.071610
     lfs-2006-05: 101552 rows after 0:01:57.967414
     lfs-2006-06: 101995 rows after 0:01:58.068903
     lfs-2006-07: 101843 rows after 0:02:00.460556
     lfs-2006-08: 102427 rows after 0:02:00.724966
     lfs-2006-09: 102951 rows after 0:01:59.257314
     lfs-2006-10: 103099 rows after 0:01:58.438681
     lfs-2006-11: 102927 rows after 0:01:57.765754
     lfs-2006-12: 102885 rows after 0:01:59.156247
     lfs-2007-01: 102578 rows after 0:01:57.116015
     lfs-2007-02: 102027 rows after 0:01:56.895181
     lfs-2007-03: 102039 rows after 0:01:53.866099
     lfs-2007-04: 102441 rows after 0:01:56.878472
     lfs-2007-05: 102553 rows after 0:01:56.003728
     lfs-2007-06: 102619 rows after 0:01:57.425871
     lfs-2007-07: 101855 rows after 0:01:55.510905
     lfs-2007-08: 102204 rows a

## Summary

We uploaded LFS public-use microdata for 2006 through 2023 into a PostgreSQL database using `SQLAlchemy` with the `psycopg` driver:
1. We used `MetaData` to create the database schema, including defining all primary and foreign keys.
2. We used the database schema to create all tables.
3. We used the `Table` definitions to load all data.
The advantage of this is that PostgreSQL enforces the constraints we put in place, ensuring that loaded rows are unique and columns with foreign key constraints do not contain values without representation in the linked table.